# Weld field inspection (RocketRide)

Upload a weld image, add **Inspector Notes** and **Blueprint Specifications**, then run the same pipeline as `app.py`:
`get_rag_context` → `detect_weld_defects` → `generate_final_report`.

Ensure a `.env` with `ROCKETRIDE_APIKEY` and `ROCKETRIDE_URI` is present, the RocketRide engine is up, and start Jupyter from this project directory (so `app.py` and the `.pipe` files load correctly).

In [ ]:
import asyncio
import os
import sys
import tempfile
from pathlib import Path

import nest_asyncio

nest_asyncio.apply()


def _ensure_project_on_path() -> Path:
    """So `import app` works when the server cwd matches this project."""
    cwd = Path.cwd().resolve()
    candidates = [cwd, cwd.parent, cwd / "rag-chatbot"]
    for c in candidates:
        if (c / "app.py").is_file():
            p = str(c)
            if p not in sys.path:
                sys.path.insert(0, p)
            return c
    raise FileNotFoundError(
        "Could not find app.py. Start Jupyter from the rag-chatbot project folder, "
        "or chdir: os.chdir('/path/to/rag-chatbot')."
    )


PROJECT_ROOT = _ensure_project_on_path()
os.chdir(PROJECT_ROOT)  # keeps relative paths to .pipe files stable


In [ ]:
import os
import tempfile
from pathlib import Path
from typing import Any, Dict, Optional, Tuple

from app import detect_weld_defects, generate_final_report, get_rag_context


async def run_field_inspection(
    image_bytes: bytes,
    image_name: str,
    inspector_notes: str,
    blueprint_specifications: str,
) -> Tuple[Optional[Dict[str, Any]], Optional[str]]:
    """
    Full orchestration: RAG (blueprint query + notes) → vision → final Markdown report.
    Returns (envelope, error_message) where `error_message` is set on failure.
    """
    bp = (blueprint_specifications or "").strip()
    if not bp:
        bp = (
            "General structural welding inspection: AWS D1.1 and ICC/IBC visual and "
            "regulatory context for field acceptance criteria and code limits."
        )
    notes = (inspector_notes or "").strip()
    ext = (Path(image_name).suffix or ".png").lower()
    if ext not in {".png", ".jpg", ".jpeg", ".gif", ".webp", ".bmp"}:
        ext = ".png"
    # Close file after write so `detect_weld_defects` can open the path (Windows-friendly).
    fd, path = tempfile.mkstemp(suffix=ext)
    try:
        try:
            os.write(fd, image_bytes)
        finally:
            os.close(fd)

        rag = await get_rag_context(bp, notes)
        if not rag.get("ok"):
            st = (rag or {}).get("error") or (rag or {}).get("stage") or str(rag)
            return None, f"RAG / code context step failed: {st}"

        vision = await detect_weld_defects(path)
        if not vision.get("ok"):
            st = (vision or {}).get("error") or (vision or {}).get("stage") or str(vision)
            return None, f"Vision (defect) step failed: {st}"

        final = await generate_final_report(rag, vision, notes)
        if not final.get("ok"):
            st = (final or {}).get("error") or (final or {}).get("stage") or str(final)
            return None, f"Report orchestration step failed: {st}"

        return final, None
    finally:
        try:
            os.unlink(path)
        except OSError:
            pass


In [ ]:
import asyncio
from pathlib import Path

import ipywidgets as w
from IPython.display import Image as IPyImage, Markdown, clear_output, display

uploader = w.FileUpload(
    accept="image/*",
    multiple=False,
    description="Upload weld image",
)

notes_area = w.Textarea(
    value="",
    placeholder="Site observations, WPS, joint ID, access constraints…",
    layout=w.Layout(width="100%", height="120px"),
)
blueprint_area = w.Textarea(
    value="",
    placeholder="Weld type, size, process, design references, local requirements…",
    layout=w.Layout(width="100%", height="120px"),
)
notes_block = w.VBox(
    [w.HTML("<b>Inspector Notes</b>"), notes_area],
    layout=w.Layout(gap="4px"),
)
blueprint_block = w.VBox(
    [w.HTML("<b>Blueprint Specifications</b>"), blueprint_area],
    layout=w.Layout(gap="4px"),
)
run_btn = w.Button(
    description="Run Inspection",
    button_style="primary",
    tooltip="Runs RAG + vision + final report (calls app.py pipeline)",
)
status = w.Output()
img_panel = w.Output(
    layout=w.Layout(
        width="45%", border="1px solid #ddd", padding="8px", min_height="240px", overflow="auto"
    )
)
report_panel = w.Output(
    layout=w.Layout(
        width="55%", border="1px solid #ddd", padding="8px", min_height="240px", overflow="auto"
    )
)
main_row = w.HBox([img_panel, report_panel], layout=w.Layout(width="100%"))


def _image_format(name: str) -> str:
    s = (Path(name).suffix or "").lower().lstrip(".")
    if s == "jpg":
        s = "jpeg"
    if s in {"png", "jpeg", "gif", "webp"}:
        return s
    return "png"


def on_run(_: w.Button) -> None:
    with status:
        clear_output()
    with img_panel:
        clear_output()
    with report_panel:
        clear_output()
    if not uploader.value:
        with status:
            print("Please upload an image first.")
        return
    fmeta = uploader.value[0]
    name = fmeta["name"]
    content: bytes = fmeta["content"]
    with status:
        print(f"Running inspection on {name!r}…")

    async def _go():
        return await run_field_inspection(
            content,
            name,
            notes_area.value,
            blueprint_area.value,
        )

    out, err = asyncio.run(_go())
    if err:
        with status:
            print(err)
        return
    assert out is not None and out.get("ok") and out.get("result") is not None
    result = out["result"]
    report_md = (result.get("field_inspection_report") or "").strip()
    if not report_md:
        with status:
            print("No report text returned (empty field_inspection_report).")
        return
    with status:
        st = (result.get("compliance_status") or "").strip()
        if st:
            print(f"Status: {st}")
        print("Done.")
    with img_panel:
        display(IPyImage(data=content, format=_image_format(name)))
    with report_panel:
        display(Markdown(report_md))


run_btn.on_click(on_run)
ui = w.VBox(
    [
        w.HTML("<h2>Weld inspection</h2>"),
        uploader,
        notes_block,
        blueprint_block,
        run_btn,
        status,
        w.HTML("<b>Image</b> (left) and <b>inspection report</b> (right)"),
        main_row,
    ],
    layout=w.Layout(gap="10px"),
)
display(ui)
